# Approach 2: Decoder-Only LoRA Fine-tuning for Bangalore Addresses

This notebook runs the complete pipeline for Approach 2:
- Decoder-only LoRA fine-tuning
- whisper-tiny base model
- Synthetic data with telephone-style corruption
- Address-focused evaluation metrics

## Step 1: Install Dependencies

In [ ]:
!pip install -q requests pandas numpy scipy python-slugify gTTS pydub soundfile torchaudio torch transformers datasets accelerate peft jiwer scikit-learn beautifulsoup4 lxml rich pyyaml

## Step 2: Upload and Extract Approach 2

In [ ]:
from google.colab import files
import zipfile
import os
from pathlib import Path

# Upload the zip file
print("Please upload 'approach_2.zip' or 'approach 2.zip' file...")
uploaded = files.upload()

# Extract approach 2
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✓ Extracted {filename}")

# Find and change to approach 2 directory
if Path('approach 2').exists():
    os.chdir('approach 2')
elif Path('approach_2').exists():
    os.chdir('approach_2')
else:
    print("⚠️  Could not find 'approach 2' directory. Listing current directory:")
    print(os.listdir('.'))
    
print(f"\n✓ Current directory: {os.getcwd()}")
print(f"✓ Files in directory: {len(os.listdir('.'))} items")

## Step 3: Generate Bangalore Address Dataset

In [ ]:
!python generate_addresses.py --skip-india-post

## Step 4: Generate Synthetic Conversations (2500 total: 40% address-heavy, 60% generic)

In [ ]:
!python generate_conversations.py --train-address-convs 800 --eval-address-convs 200 --non-address-ratio 1.5

## Step 5: Generate Audio with Telephone-Style Corruption

⚠️ **This step takes the longest** (~1-2 hours) due to gTTS API calls.

Audio will have:
- Band-pass filter (300-3400 Hz)
- Speed jitter (±10%)
- Occasional clipping (10% chance)

In [ ]:
!python generate_audio.py

## Step 6: Prepare Whisper Dataset

In [ ]:
!python prepare_whisper_dataset.py

## Step 7: Fine-tune Whisper with Decoder-Only LoRA

Configuration:
- Base model: `openai/whisper-tiny`
- LoRA: r=32, alpha=64, dropout=0.05
- Target: Decoder attention layers only (encoder frozen)
- Learning rate: 5e-5
- Epochs: Up to 10 (with early stopping)

⏱️ **Training takes ~30-60 minutes** on T4 GPU

In [ ]:
!python finetune_whisper.py \
  --model-name openai/whisper-tiny \
  --num-train-epochs 10 \
  --learning-rate 5e-5 \
  --per-device-train-batch-size 8 \
  --output-dir ./models/whisper_bangalore_lora

## Step 8: Evaluate Model

Metrics:
1. **Exact canonical address accuracy** (primary)
2. **Address-token F1 score** (primary)
3. **Overall WER** (regression check only)

In [ ]:
!python evaluate_whisper.py \
  --base-model-name openai/whisper-tiny \
  --lora-dir ./models/whisper_bangalore_lora \
  --max-samples 500

## Step 9: Download Trained Model (Optional)

Download the LoRA adapter for use elsewhere:

In [ ]:
from google.colab import files
import shutil

# Create a zip of the model
model_dir = './models/whisper_bangalore_lora'
if os.path.exists(model_dir):
    shutil.make_archive('whisper_bangalore_lora', 'zip', model_dir)
    files.download('whisper_bangalore_lora.zip')
    print("✓ Model downloaded!")
else:
    print("⚠️  Model directory not found. Make sure training completed successfully.")